# 03 — Data Preprocessing & Feature Engineering

## Objective
Prepare the Bank Marketing dataset for machine learning by investigating
duplicate records, handling unknown values, reviewing feature types,
performing feature engineering, and defining a leakage-safe preprocessing
strategy.

## Key modelling considerations
- Target variable: `y`
- Binary classification: yes / no
- Exclude `duration` from the primary model due to data leakage.
- Preserve the original raw dataset.
- Fit preprocessing transformations only on training data.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "bank-additional-full.csv"

df = pd.read_csv(data_path, sep=";")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no


In [3]:
df_work = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_work.shape)

Original dataset shape: (41188, 21)
Working dataset shape: (41188, 21)


## 1. Duplicate Record Investigation

During EDA, 12 duplicate records were identified.
Before removing them, we will inspect the duplicate observations
to determine whether they are exact duplicates.

In [4]:
duplicate_count = df_work.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 12


In [5]:
duplicate_rows = df_work[df_work.duplicated(keep=False)]

print("Total rows involved in duplication:", len(duplicate_rows))

duplicate_rows.sort_values(
    by=list(df_work.columns)
).head(30)

Total rows involved in duplication: 24


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
28476,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.80,93.08,-47.10,1.42,5099.10,no
28477,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.80,93.08,-47.10,1.42,5099.10,no
14155,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.40,93.92,-42.70,4.96,5228.10,no
14234,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.40,93.92,-42.70,4.96,5228.10,no
18464,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
18465,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
32505,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.80,92.89,-46.20,1.31,5099.10,no
32516,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.80,92.89,-46.20,1.31,5099.10,no
12260,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
12261,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no


## 2. Removing Duplicate Records

The initial dataset contained 12 duplicate occurrences,
representing 12 pairs of exact duplicate records.

Since these records were identical across all available columns,
we retain the first occurrence and remove the repeated copies.

The original dataset remains unchanged.

In [6]:
# Remove exact duplicate records
df_work = df_work.drop_duplicates(keep="first").copy()

print("Shape after removing duplicates:", df_work.shape)
print("Remaining duplicate records:", df_work.duplicated().sum())

Shape after removing duplicates: (41176, 21)
Remaining duplicate records: 0


In [7]:
print("Original dataset shape:", df.shape)
print("Cleaned working dataset shape:", df_work.shape)

Original dataset shape: (41188, 21)
Cleaned working dataset shape: (41176, 21)


## 3. Investigation of Unknown Values

The dataset contains unknown categories in several categorical
features. We will examine their frequency and relationship with
the target variable before deciding how to handle them.

In [8]:
# Identify categorical columns
categorical_columns = df_work.select_dtypes(
    include=["object", "str"]
).columns

# Count unknown values in each categorical column
unknown_counts = (
    df_work[categorical_columns]
    .eq("unknown")
    .sum()
)

# Calculate percentages
unknown_percentage = (
    unknown_counts / len(df_work) * 100
)

unknown_summary = pd.DataFrame({
    "unknown_count": unknown_counts,
    "unknown_percentage": unknown_percentage
})

unknown_summary = unknown_summary[
    unknown_summary["unknown_count"] > 0
].sort_values(
    by="unknown_count",
    ascending=False
)

unknown_summary.round(2)

,unknown_count,unknown_percentage
default,8596,20.88
education,1730,4.20
housing,990,2.40
loan,990,2.40
job,330,0.80
marital,80,0.19


In [9]:
unknown_target_analysis = {}

for col in unknown_summary.index:

    analysis = df_work.groupby(
        df_work[col].eq("unknown").map({
            True: "unknown",
            False: "known"
        }),
        observed=True
    )["y"].agg(
        total_customers="count",
        subscriptions=lambda x: (x == "yes").sum()
    )

    analysis["subscription_rate"] = (
        analysis["subscriptions"] /
        analysis["total_customers"] * 100
    )

    unknown_target_analysis[col] = analysis.round(2)

for col, result in unknown_target_analysis.items():
    print(f"\nSubscription analysis: {col}")
    display(result)


Subscription analysis: default


,total_customers,subscriptions,subscription_rate
default,,,
known,32580,4196,12.88
unknown,8596,443,5.15



Subscription analysis: education


,total_customers,subscriptions,subscription_rate
education,,,
known,39446,4388,11.12
unknown,1730,251,14.51



Subscription analysis: housing


,total_customers,subscriptions,subscription_rate
housing,,,
known,40186,4532,11.28
unknown,990,107,10.81


... (2 outputs left out)


Subscription analysis: job


,total_customers,subscriptions,subscription_rate
job,,,
known,40846,4602,11.27
unknown,330,37,11.21



Subscription analysis: marital


,total_customers,subscriptions,subscription_rate
marital,,,
known,41096,4627,11.26
unknown,80,12,15.00


### 3.1 Decision: Handling Unknown Categories

Unknown values were found in six categorical features:
default, education, housing, loan, job and marital.

Their subscription rates were compared against records with
known information.

The unknown categories will be retained rather than replaced
with the most frequent category or removed.

This preserves the distinction between unavailable information
and an explicitly recorded category.

The marital unknown category has a relatively small sample size,
so its observed subscription rate should be interpreted cautiously.

## 4. Investigation of Special Values

The pdays feature uses 999 as a special indicator for customers
who were not previously contacted.

Similarly, poutcome contains the category 'nonexistent' for
customers without a previous campaign outcome.

These values represent meaningful business information and
will not be treated as ordinary missing values.

In [10]:
print("pdays value distribution:")
display(df_work["pdays"].value_counts().head(10))

print("\npoutcome value distribution:")
display(df_work["poutcome"].value_counts())

pdays value distribution:


pdays
999    39661
3        439
6        412
4        118
9         64
2         61
7         60
12        58
10        52
5         46
Name: count, dtype: int64


poutcome value distribution:


poutcome
nonexistent    35551
failure         4252
success         1373
Name: count, dtype: int64

In [11]:
missing_summary = pd.DataFrame({
    "missing_count": df_work.isnull().sum(),
    "missing_percentage": (
        df_work.isnull().mean() * 100
    )
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,missing_count,missing_percentage


### 4.1 pdays

The pdays feature represents the number of days since the
customer was last contacted during a previous campaign.

A value of 999 indicates that the customer was not previously
contacted. This is a special indicator rather than an actual
elapsed duration of 999 days.

In [12]:
print("pdays distribution:")
display(
    df_work["pdays"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nNumber of records with pdays = 999:")
print((df_work["pdays"] == 999).sum())

print("\nPercentage of records with pdays = 999:")
print(round((df_work["pdays"] == 999).mean() * 100, 2), "%")

pdays distribution:


pdays
0      15
1      26
2      61
3     439
4     118
5      46
6     412
7      60
8      18
9      64
10     52
11     28
12     58
13     36
14     20
15     24
16     11
17      8
18      7
19      3
Name: count, dtype: int64


Number of records with pdays = 999:
39661

Percentage of records with pdays = 999:
96.32 %


In [13]:
print("poutcome distribution:")

display(
    df_work["poutcome"].value_counts()
)

poutcome distribution:


poutcome
nonexistent    35551
failure         4252
success         1373
Name: count, dtype: int64